# 🔥 Análisis de Severidad de Incendio 🔥

Este notebook corre **GRASS dentro de Google Colab** usando la API
`grass.script` — compatible con las versiones disponibles en Ubuntu 22.04.
La visualización usa `grass.jupyter`.

---
## 1. Instalación de GRASS en Google Colab

Instalamos GRASS desde el repositorio `ppa:ubuntugis/ubuntugis-unstable`. Esto puede tardar 2-3 minutos.

In [ ]:
# Agregar repositorio UbuntuGIS e instalar GRASS
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt-get update -qq
!apt-get install -y -qq grass-core grass-dev

# Verificar versión instalada
!grass --version

---
## 2. Configuración del entorno Python

Le indicamos a Python dónde encontrar los paquetes de GRASS e importamos las bibliotecas necesarias:
- **`grass.script`** (`gs`): gestión de proyectos y sesiones
- **`grass.jupyter`** (`gj`): visualización interactiva de mapas en notebooks

In [ ]:
import sys
import subprocess

# Agregar los paquetes Python de GRASS al path
grass_python_path = subprocess.check_output(
    ["grass", "--config", "python_path"], text=True
).strip()
sys.path.append(grass_python_path)
print(f"GRASS Python path: {grass_python_path}")

In [ ]:
# Importar las bibliotecas de GRASS
import grass.script as gs
import grass.jupyter as gj

print("✅ grass.script y grass.jupyter importados")

---
## 3. Acceso a los datos desde Google Drive

Para asegurar que los datos persistan, trabajaremos directamente desde tu unidad de Google Drive.

1. Asegúrense de haber copiado la carpeta "Jornadas 360 UNC" a su propia unidad de Drive (Mi unidad).
2. Ejecuten la celda de abajo para montar su Drive en Colab y verificar la ruta de los archivos.

In [ ]:
from google.colab import drive
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta a la carpeta en tu unidad
DRIVE_FOLDER = "/content/drive/MyDrive/Jornadas360UNC"

# Definimos las rutas para los dos stacks (Pre y Post)
PATH_PRE  = os.path.join(DRIVE_FOLDER, "S2_PNAlerces_pre.tif")
PATH_POST = os.path.join(DRIVE_FOLDER, "S2_PNAlerces_post.tif")

# Verificación de los archivos
if os.path.exists(PATH_PRE) and os.path.exists(PATH_POST):
    print(f"✅ Archivos detectados con éxito en: {DRIVE_FOLDER}")
    print(f"   - Pre-fuego: {os.path.basename(PATH_PRE)}")
    print(f"   - Post-fuego: {os.path.basename(PATH_POST)}")
else:
    print(f"❌ Error: No se encontraron los archivos en {DRIVE_FOLDER}")
    print("Por favor, verificá que la carpeta 'Jornadas360UNC' esté en la raíz de tu 'Mi unidad'.")

---
## 4. Crear el proyecto GRASS e iniciar la sesión

En este paso vamos a crear nuestro espacio de trabajo de GRASS. El proyecto se guardará físicamente en tu Google Drive dentro de la carpeta del taller, usando la configuración de coordenadas del archivo original. Luego iniciamos la sesión con `gj.init()`.

In [ ]:
# 1. Definir la ruta del proyecto (se guardará dentro de 'Jornadas360UNC')
PROJECT_PATH = os.path.join(DRIVE_FOLDER, "PNAlerces_fuego")
print(PROJECT_PATH)

# 2. Crear el proyecto GRASS
# Usamos el stack pre-fuego para que el proyecto herede automáticamente el sistema de coordenadas
gs.create_project(path=PROJECT_PATH, filename=PATH_PRE)

# 3. Iniciar la sesión de trabajo
session = gj.init(PROJECT_PATH)

---
## 5. Importar los datos pre y post fuego

Importamos los stacks pre y post fuego con `r.in.gdal`. GRASS crea un raster por banda: `stack.1` … `stack.6`. Para refrescar la memoria, acá está el detalle:

| Nombre en GRASS | Banda Sentinel-2 | Descripción | Uso en el Análisis |
| :--- | :--- | :--- | :--- |
| `nombre.1` | **B2** | Azul (Visible) | - |
| `nombre.2` | **B3** | Verde (Visible) | - |
| `nombre.3` | **B4** | Rojo (Visible) | Cálculo de NDVI |
| `nombre.4` | **B8** | NIR (Infrarrojo Cercano) | Cálculo de NDVI y NBR |
| `nombre.5` | **B11** | SWIR 1 (Infrarrojo de Onda Corta) | Enmascaramiento de agua |
| `nombre.6` | **B12** | SWIR 2 (Infrarrojo de Onda Corta) | Cálculo de NBR |

In [ ]:
# 1. Importar el stack pre-fuego
gs.run_command("r.in.gdal",
               input=PATH_PRE,
               output="pre",
               memory=2000)

In [ ]:
# 2. Importar el stack post-fuego
gs.run_command("r.in.gdal",
               input=PATH_POST,
               output="post",
               memory=2000)

In [ ]:
# 3. Verificar la lista de mapas importados
print("✅ Capas cargadas en el proyecto:")
lista_mapas = gs.list_pairs(type="raster")
for nombre, archivo in lista_mapas:
    print(f" - {nombre}")

---
## 6. Definir la región de trabajo

Ajustamos la región computacional al extent y resolución de una de las bandas importadas.

In [ ]:
# Ajustar la región al raster stack.1
gs.run_command("g.region", raster="pre.1")

# Imprimir la región (read_command devuelve texto como string)
print(gs.read_command("g.region", flags="p"))

---
## 7. Álgebra de mapas — Cálculo de NDVI y NBR

| Índice | Fórmula | Bandas |
|--------|---------|--------|
| NDVI_pre | (NIR − Rojo) / (NIR + Rojo) | pre.4, pre.3 |
| NDVI_post | (NIR − Rojo) / (NIR + Rojo) | post.4, post.3 |
| NBR_pre  | (NIR − SWIR2) / (NIR + SWIR2) | pre.4, pre.6 |
| NBR_post | (NIR − SWIR2) / (NIR + SWIR2) | post.4, post.6 |

In [ ]:
# NDVI pre-incendio
gs.mapcalc(
    "ndvi_pre = float(pre.4 - pre.3) / (pre.4 + pre.3)",
    overwrite=True
)
print("✅ ndvi_pre")

# NDVI post-incendio
gs.mapcalc(
    "ndvi_post = float(post.4 - post.3) / (post.4 + post.3)",
    overwrite=True
)
print("✅ ndvi_post")

In [ ]:
# NBR pre-incendio
gs.mapcalc(
    "nbr_pre = float(pre.4 - pre.6) / (pre.4 + pre.6)",
    overwrite=True
)
print("✅ nbr_pre")

# NBR post-incendio
gs.mapcalc(
    "nbr_post = float(post.4 - post.6) / (post.4 + post.6)",
    overwrite=True
)
print("✅ nbr_post")

In [ ]:
# Tablas de color predefinidas (run_command: sin salida que necesitemos)
gs.run_command("r.colors", map="ndvi_pre", color="ndvi")
gs.run_command("r.colors", map="ndvi_post", color="ndvi")
gs.run_command("r.colors", map="nbr_pre",  color="differences")
gs.run_command("r.colors", map="nbr_post", color="differences")
print("\n✅ Tablas de color asignadas")

---
## 8. dNBR y máscaras de severidad

- **dNBR** = NBR_pre − NBR_post
- **tierra** = máscara que excluye agua (SWIR1_pre > 500)
- **sev_alta** = dNBR > 0.44 sobre tierra (Key & Benson 2006)

Para paletas de color personalizadas usamos `gs.write_command()`,
que envía texto como `stdin` — equivalente a pasarle un archivo de reglas.

In [ ]:
# dNBR
gs.mapcalc("dnbr = nbr_pre - nbr_post", overwrite=True)
print("✅ dnbr")

# Máscara de tierra (excluye cuerpos de agua con SWIR1 bajo en el stack pre fuego)
gs.mapcalc("tierra = if(pre.5 > 500, 1, null())", overwrite=True)
print("✅ tierra")

# Severidad alta
gs.mapcalc("sev_alta = if(tierra && dnbr > 0.44, 1, null())", overwrite=True)
print("✅ sev_alta")

---
## 9. Reporte de superficie afectada


In [ ]:
# read_command devuelve el texto de salida como string
reporte = gs.read_command("r.report", map="sev_alta", units="h")
print(reporte)

---
## 10. Visualización interactiva

Usamos `grass.jupyter.InteractiveMap` para explorar los mapas en el notebook,
y `grass.jupyter.Map` para una figura estática con leyenda exportable.

In [ ]:
# NDVI pre-incendio
print("NDVI pre-incendio")
m = gj.InteractiveMap(width=700)
m.add_raster("ndvi_pre", opacity=0.85)
m.add_layer_control()
m.show()

In [ ]:
# Severidad alta — mapa interactivo
print("Severidad alta")
m = gj.InteractiveMap(width=700)
m.add_raster("sev_alta")
m.add_layer_control()
m.show()

In [ ]:
# Vista estática con leyenda — exportable como PNG
# Map() renderiza usando la región computacional activa
fig = gj.Map(width=600, use_region=True)
fig.d_rast(map="ndvi_pre")
fig.d_rast(map="sev_alta")
fig.d_text(
    text="Severidad de incendio — Alerces, Cholila 2026",
    at=(50, 97),
    align="cc",
    size=2.5
)
fig.show()

# Para guardar como PNG:
# fig.save("/content/severidad_alerces.png")

---
## Referencias

- **GRASS Development Team (2024)**. GRASS GIS. https://grass.osgeo.org
- **Andreo, V. (2024)**. Get started with GRASS in Google Colab. https://grass-tutorials.osgeo.org
- **Andreo, V. (2024)**. Get started with GRASS & Python in Jupyter Notebooks. https://grass-tutorials.osgeo.org